# 🧭 3. The Navigator: 웹 브라우징 도구 체계와 에이전트 탐색

웹 공간은 전 세계에서 가장 거대하고 동적인 데이터의 바다입니다.
하지만 웹사이트마다 제각각인 HTML 구조, 복잡한 JavaScript 렌더링, 로그인 및 봇 차단 솔루션 때문에
AI 에이전트가 웹에서 원하는 데이터를 스스로 찾아 수집하는 것은 결코 쉬운 일이 아닙니다.

이번 실습에서는 **AI 에이전트에게 웹 브라우저라는 '눈과 손발'을 부여하는 방법**과,
토큰과 시간을 획기적으로 아끼는 **3단계 에스컬레이션 브라우징 도구 체계(Level 1 / Level 2 / Level 3)**를 직접 체험하고 구축합니다.

---

### 🎓 학습 목차 (Curriculum Flow)

| 파트 | 주제 | 핵심 내용 |
|:---:|:---|:---|
| **Part 1** | **웹 데이터 수집 개론 & 핵심 인프라** | HTTP vs 브라우저 자동화, Chromium, Playwright, Headless vs Headful, noVNC 가상 화면 |
| **Part 2** | **Level 1: 정적/경량 탐색 도구 & 에이전트** | DOM Skeleton 맵, Scoped Section 파싱, Selector 검증, 에이전트 분석 & 스크래핑 파이프라인 |
| **Part 3** | **Level 2: 경량 인터랙션 & noVNC 실시간 시청** | `interact_page`, AJAX 렌더링 대기, 더보기 클릭, `take_screenshot` 캡처 |
| **Part 4** | **Level 3: 자율 에이전트 (`browse_web`) & 세션 공유** | `browser-use` Agent-as-a-Tool 연동, 로그인 자율 탐색, CDP 브라우저 세션 공유 |

---

## 🛠️ Step 0. 환경 세팅

필요한 패키지를 로드하고 프로젝트 루트 경로 및 비동기 이벤트 루프(`nest_asyncio`)를 설정합니다.

In [ ]:
import os
import sys
import asyncio
import nest_asyncio
from dotenv import load_dotenv

# 1. 환경변수 로드
load_dotenv(override=True)

# 2. 프로젝트 루트 경로 자동 설정 (상위 탐색)
project_root = os.getcwd()
for _ in range(5):
    if os.path.exists(os.path.join(project_root, "app")):
        break
    project_root = os.path.dirname(project_root)

os.chdir(project_root)
if project_root not in sys.path:
    sys.path.insert(0, project_root)

print(f"✅ Working Directory: {os.getcwd()}")
print(f"✅ Project Root: {project_root}")

# 3. 주피터 노트북 비동기 루프 중복 방지
nest_asyncio.apply()

from app.utils import normalize_content

In [ ]:
os.environ['HEADLESS'] = "false"

---
## 📖 Part 1. 웹 데이터 수집 개론 & 핵심 인프라

웹에서 데이터를 수집하는 방식은 크게 **두 가지 접근법**으로 나뉩니다.

```
┌─────────────────────────────────────────────────────────────────────────────┐
│                        웹 데이터 수집의 두 가지 경로                         │
├──────────────────────────────────────┬──────────────────────────────────────┤
│      [경로 A] HTTP 패킷 요청          │       [경로 B] 브라우저 자동화       │
│   (requests, httpx, BeautifulSoup)   │        (Playwright, Selenium)        │
├──────────────────────────────────────┼──────────────────────────────────────┤
│ 🌐 URL ➔ HTML 텍스트 다운로드        │ 🖥️ 실제 Chromium 브라우저 기동       │
│ ⚡ 속도: 0.1~0.5초 (초고속)          │ ⏱️ 속도: 2~5초 (상대적으로 무거움)    │
│ 💰 리소스: 메모리 수 MB 소모         │ 🏋️ 리소스: 수백 MB~1GB 메모리 소모   │
│ ❌ JavaScript 실행 불가 (동적 웹 X)  │ ⭕ JavaScript 완벽 실행 (React, Vue) │
│ ❌ 버튼 클릭, 폼 입력, 스크롤 불가   │ ⭕ 클릭, 스크롤, 로그인, 팝업 제어   │
└──────────────────────────────────────┴──────────────────────────────────────┘
```

### 1.1 HTTP 요청 vs 브라우저 자동화 비교

| 비교 항목 | HTTP 요청 (requests, BeautifulSoup) | 브라우저 자동화 (Playwright) |
| :--- | :--- | :--- |
| **작동 원리** | 서버에 GET/POST 요청을 직접 보내 원시 HTML 텍스트 수집 | 실제 브라우저 엔진을 프로세스로 띄워 물리적으로 페이지 렌더링 |
| **JavaScript 렌더링** | ❌ 불가 (React/Next.js/Vue 등 SPA는 빈 껍데기만 수집) | ⭕ 완벽 지원 (클라이언트 측 렌더링 및 AJAX 지연 데이터 확보) |
| **사용자 인터랙션** | ❌ 불가능 (스크롤, 버튼 클릭, 로그인 입력 불가) | ⭕ 마우스 클릭, 키보드 입력, 무한 스크롤, 탭 전환 완벽 제어 |
| **실행 속도 & 비용** | ⚡ 초고속 (0.1~0.5초), CPU/메모리 소모 극히 미미 | 🐢 상대적으로 무거움 (1~3초), 브라우저 프로세스 리소스 소모 |
| **최적 활용 영역** | 정적 사이트, 공공 API, 대량 배치 수집 (100~10,000건) | 사이트 구조 분석, 동적 UI 조작, 로그인, 봇 방지 우회 |

---

### 1.2 핵심 용어 및 인프라 개념

AI 에이전트가 웹 브라우징을 수행할 때 반드시 알아야 할 핵심 인프라 개념들입니다.

| 인프라 / 기술 | 핵심 정의 및 역할 |
| :--- | :--- |
| **Chromium** | Chrome, Edge, Brave의 모태가 되는 오픈소스 브라우저 엔진. 에이전트의 **'물리적 손과 발'** 역할을 합니다. |
| **Playwright** | Microsoft에서 개발한 현대적 브라우저 자동화 도구. 요소 자동 대기(**Auto-waiting**)와 빠른 속도를 자랑합니다. |
| **Headless 모드** | 화면(GUI 창)을 모니터에 띄우지 않고 메모리 상에서만 브라우저를 구동하는 모드. 서버 배포 및 리소스 절약에 필수적입니다. |
| **Headful 모드** | 사람이 직접 눈으로 볼 수 있는 실제 창 형태로 브라우저를 띄우는 모드. **동작 관찰 및 디버깅**에 사용됩니다. |
| **Xvfb (가상 디스플레이)** | 리눅스/WSL 환경에서 물리적 모니터 없이 메모리 상에 가상 모니터(`DISPLAY=:99`)를 생성하여 크래시를 방지합니다. |
| **noVNC** | 가상 디스플레이 화면을 웹 브라우저(포트 `6080`)로 실시간 스트리밍해주는 뷰어. 에이전트의 브라우저 조작을 생중계로 봅니다. |

---

### 1.3 왜 3단계 에스컬레이션 도구 체계인가?

에이전트가 모든 웹 작업을 무거운 자율 브라우징(`browser-use`)으로만 처리하면 **속도가 수십 배 느려지고 LLM 토큰 비용이 폭발**합니다.
따라서 우리는 **가장 가볍고 빠른 도구부터 시작하여 필요할 때만 상위 단계로 에스컬레이션**하는 전략을 사용합니다.

```
┌─────────────────────────────────────────────────────────────────────────────┐
│                      3단계 브라우징 에스컬레이션 피라미드                   │
├─────────────────────────────────────────────────────────────────────────────┤
│  Level 3: 자율 에이전트 (browse_web) ➔ 10~30초                              │
│    - 비전(스크린샷) 기반 판단, 로그인/CAPTCHA, 고난이도 자율 탐색           │
│    ──────────────────────────────────────────────────────────────────────── │
│  Level 2: 경량 인터랙션 (interact_page, take_screenshot) ➔ 1~3초            │
│    - 더보기 클릭, 탭 전환, 검색어 입력, AJAX 렌더링 대기                   │
│    ──────────────────────────────────────────────────────────────────────── │
│  Level 1: 정적/경량 탐색 (skeleton, section, verify) ➔ 0.5~1초              │
│    - DOM 구조 파악, 영역별 HTML 분석, CSS 셀렉터 검증 (대부분 여기서 해결)  │
└─────────────────────────────────────────────────────────────────────────────┘
```

---
## 🔍 Part 2. Level 1: 정적/경량 탐색 도구 & 에이전트

Level 1 도구들은 웹페이지 전체 HTML(수만 줄)을 LLM에게 그대로 넘기는 대신,
**핵심 뼈대만 요약하거나 필요한 영역만 정밀 추출**하여 토큰을 90% 이상 절약합니다.

### 2.1 Level 1 도구 3총사 직접 사용해보기
1. `extract_dom_skeleton`: 웹페이지의 DOM 구조를 태그 트리 맵으로 요약 (텍스트 축약)
2. `get_page_section`: 특정 CSS 셀렉터 영역의 HTML만 떼어내어 정밀 분석
3. `verify_selectors`: 에이전트가 추론한 셀렉터가 실제로 데이터를 뽑아내는지 1초 만에 검증

In [ ]:
from app.tools.navigator import (
    extract_dom_skeleton,
    get_page_section,
    verify_selectors
)

TARGET_URL = "http://quotes.toscrape.com"

# 1. DOM Skeleton 추출: 웹사이트의 전체 레이아웃 뼈대 확인
print("=== [1] DOM Skeleton 추출 ===")
skeleton_result = await extract_dom_skeleton.ainvoke({
    "url": TARGET_URL,
    "root_selector": "body",
    "max_depth": 4,
    "wait_ms": 1000
})
print(skeleton_result[:800])
print("\n... (중략) ...\n")

In [ ]:
# 2. Page Section 추출: 인용구 카드(.quote) 영역만 스코핑하여 정밀 확인
print("=== [2] Page Section 추출 (.quote) ===")
section_result = await get_page_section.ainvoke({
    "url": TARGET_URL,
    "root_selector": "div.quote",
    "max_chars": 1500
})
print(section_result)

# 3. Verify Selectors 검증: text, author, tag 셀렉터가 실제 매칭되는지 확인
print("\n=== [3] Selectors 검증 ===")
verify_result = await verify_selectors.ainvoke({
    "url": TARGET_URL,
    "selectors_json": '{"text": "div.quote span.text", "author": "div.quote small.author", "tags": "div.quote div.tags a.tag"}'
})
print(verify_result)

### 2.2 Level 1 도구를 장착한 Scraper 에이전트 구축 및 분석

이제 위 Level 1 도구들을 장착한 **Scraper 에이전트**에게 사이트 분석을 지시해 보겠습니다.
에이전트는 스스로 도구를 호출하여 DOM 구조를 파악하고, 최적의 CSS 셀렉터와 URL 패턴을 보고서로 작성합니다.

In [ ]:
from app.agents.scraper import create_agent_executor
import uuid

# 1. Scraper 에이전트 팩토리로부터 에이전트 인스턴스 생성
scraper_agent = await create_agent_executor()

# 2. 대화 세션 및 체크포인터 메모리 관리를 위한 config 설정 (thread_id 필수)
thread_id = f"nav_{uuid.uuid4().hex[:8]}"
config = {"configurable": {"thread_id": thread_id}}
print(f"📌 세션 ID: {thread_id}")

# 3. 1단계 미션: 사이트 구조 탐색 및 데이터 패턴 분석 지시
mission_analyze = """
http://quotes.toscrape.com 사이트를 방문하여 다음을 수행하세요:
1. extract_dom_skeleton과 get_page_section을 사용하여 인용구 목록의 구조를 파악하세요.
2. 인용구 내용(text), 저자(author), 태그 목록(tags)을 추출할 수 있는 정확한 CSS 셀렉터를 찾아 verify_selectors로 검증하세요.
3. 페이지네이션(다중 페이지 이동) URL 패턴을 확인하고, 분석 결과를 일목요연하게 보고하세요.
(주의: 지금은 크롤링 코드를 작성하지 말고 분석 보고서만 제출하세요)
"""

print("🚀 [Scraper Agent] 사이트 구조 분석 시작...\n")
response = await scraper_agent.ainvoke(
    {"messages": [("user", mission_analyze)]},
    config=config
)



In [ ]:
from app.utils.message_utils import normalize_content

# 에이전트의 최종 분석 보고서 출력
print(normalize_content(response["messages"][-1].content))

### 2.3 분석 결과를 바탕으로 실제 스크래핑 코드 작성 및 수집

분석이 끝났으면, 이제 에이전트에게 **빠르고 가벼운 Python 수집 코드(`requests` + `BeautifulSoup`)**를 작성하고 실행하도록 지시합니다.
무거운 브라우저 대신 HTTP 요청 루프를 통해 1~3페이지의 인용구 30건을 단 1초 만에 수집합니다.

In [ ]:
output_json_path = os.path.abspath("./artifacts/notebooks/generated/quotes_sample_result.json")
os.makedirs(os.path.dirname(output_json_path), exist_ok=True)

mission_crawl = f"""
앞서 분석한 셀렉터와 URL 패턴을 기반으로:
1. 1페이지부터 3페이지까지 모든 인용구를 수집하는 가벼운 Python 스크립트(requests + BeautifulSoup)를 작성하세요 (file_writer).
2. 작성한 스크립트를 실행하여 (bash_command) 다음 경로에 JSON 파일로 저장하세요:
   저장 경로: {output_json_path}
3. 저장이 완료되면 수집된 건수와 샘플 데이터를 확인하여 최종 보고서를 작성하세요.
"""

# 동일한 thread_id를 전달하여 앞서 분석한 대화 맥락을 그대로 이어받아 수집 지시
print("🚀 [Scraper Agent] 스크래핑 코드 작성 및 실행 시작...\n")
response_crawl = await scraper_agent.ainvoke(
    {"messages": [("user", mission_crawl)]},
    config=config
)

In [ ]:
# 에이전트의 최종 분석 보고서 출력
print(normalize_content(response["messages"][-1].content))

In [ ]:
# 수집된 JSON 파일 확인
import json

if os.path.exists(output_json_path):
    with open(output_json_path, "r", encoding="utf-8") as f:
        data = json.load(f)
    print(f"✅ 수집 성공! 총 {len(data)}건의 인용구 수집 완료\n")
    print("--- 상위 2건 샘플 ---")
    print(json.dumps(data[:2], indent=2, ensure_ascii=False))
else:
    print("❌ 결과 파일이 생성되지 않았습니다.")

---
## 🖱️ Part 3. Level 2: 경량 인터랙션 도구 & noVNC 실시간 시청

정적 분석(Level 1)만으로는 해결할 수 없는 웹사이트들이 있습니다:
- 🔍 검색어를 입력하고 엔터를 눌러야 결과가 나오는 사이트
- 📜 '더보기(Load More)' 버튼을 클릭해야 추가 데이터가 렌더링되는 사이트
- ⏳ AJAX 비동기 호출로 2~3초 뒤에 데이터가 늦게 로딩되는 사이트

이럴 때 사용하는 도구가 **`interact_page`**와 **`take_screenshot`**입니다.

---

### 📺 브라우저 화면 시청 가이드

에이전트가 브라우저를 조작하는 모습을 실시간으로 보려면 `.env`에서 **두 가지 설정**이 필요합니다:

```bash
# .env 파일
HEADLESS=false        # headed 모드로 전환 (브라우저 창이 화면에 표시됨)

# [선택] Codespaces/원격 환경에서 noVNC 사용 시:
DISPLAY=":1"          # 가상 디스플레이 번호 지정
```

| 환경 | 설정 | 결과 |
| :--- | :--- | :--- |
| **Headless** (기본) | `HEADLESS=true` | 화면 없이 백그라운드 실행 |
| **로컬 데스크톱** | `HEADLESS=false` | 내 화면에서 브라우저 창 직접 시청 |
| **Codespaces + noVNC** | `HEADLESS=false` + `DISPLAY=":1"` | noVNC 웹 뷰어에서 시청 |

**noVNC로 시청하는 경우:**
1. 웹 브라우저 새 탭을 열고 `http://localhost:6080/vnc.html`에 접속합니다.
2. **'Connect'** 버튼을 클릭하면 가상 디스플레이 화면이 나타납니다.
3. 아래 셀들을 실행하면 **브라우저가 자동으로 열려 클릭/입력하는 모습**을 실시간으로 시청할 수 있습니다!

> ⚠️ **참고**: 현재 `HEADLESS=true` (기본값)이면 브라우저 창이 보이지 않지만 도구는 정상 동작합니다.  
> 화면을 보지 않아도 실습 진행에는 문제가 없습니다.

In [ ]:
from app.tools.navigator import interact_page, take_screenshot
from IPython.display import Image, display
import json

# 시나리오: Quotes to Scrape의 태그 필터 인터랙션
# 태그 중 'inspirational' 태그를 클릭하여 해당 카테고리로 필터링 이동

print("=== [Level 2] interact_page: 태그 클릭 인터랙션 ===")
interact_result = await interact_page.ainvoke({
    "url": "http://quotes.toscrape.com",
    "actions_json": json.dumps([
        {"action": "click", "selector": "a[href='/tag/inspirational/']"},
        {"action": "wait", "selector": "div.quote", "timeout_ms": 5000}
    ]),
    "wait_ms": 2000
})
print(interact_result)

In [ ]:
# take_screenshot으로 필터링된 결과 화면을 캡처하여 인라인 확인
screenshot_path = await take_screenshot.ainvoke({
    "url": "http://quotes.toscrape.com/tag/inspirational/",
    "filename": "quotes_inspirational_demo.png",
    "full_page": False
})
print(screenshot_path)

# 캡처된 이미지 인라인 렌더링
img_file = "./artifacts/screenshots/quotes_inspirational_demo.png"
if os.path.exists(img_file):
    display(Image(filename=img_file, width=700))
else:
    print("스크린샷 파일 경로를 확인하세요.")

---
## 🤖 Part 4. Level 3: 자율 브라우저 에이전트 (`browse_web`) & 세션 공유

Level 1과 Level 2로도 해결하기 힘든 최후의 난관들이 있습니다:
- 🔐 복잡한 다단계 로그인 / 2FA
- 🧩 난독화된 DOM (무작위 class 이름으로 매번 바뀌는 현대적 SPA)
- 🛡️ CAPTCHA / 봇 탐지 챌린지
- 🗺️ 어디를 눌러야 할지 모르는 미지의 UI에서 시각적 판단이 필요할 때

이때는 **비전(Vision) 기반의 자율 에이전트인 `browser-use`**를 **Agent-as-a-Tool** 방식으로 호출합니다.

Browser-use(공식 GitHub: [browser-use](https://github.com/browser-use/browser-use))는 에이전트가 사람처럼 웹 브라우저를 직접 제어할 수 있게 해주는 파이썬 라이브러리입니다. 단순히 데이터를 긁어오는 '크롤링'을 넘어, AI가 웹사이트의 버튼을 클릭하고, 정보를 검색하고, 복잡한 워크플로우를 스스로 수행하게 만드는 웹 에이전트(Web Agent) 개발의 핵심 도구입니다.

### 주요 기능

- **지능적 브라우징**: 정해진 스크립트가 아니라, AI가 화면을 보고 다음 행동(클릭, 입력, 스크롤 등)을 스스로 판단합니다.
- **Playwright 기반**: 강력한 브라우저 자동화 도구인 Playwright를 사용하여 Chrome, Firefox 등 다양한 브라우저를 제어합니다.
- **간편한 통합**: LangChain과 같은 프레임워크와 잘 연동되며, Google, OpenAI, Anthropic의 최신 모델을 비전(Vision) 기능을 통해 활용합니다.
- **멀티 탭 지원**: 여러 탭을 동시에 관리하며 복잡한 정보를 비교하거나 수집할 수 있습니다.

### 작동 원리

- **관찰(Observe)**: 현재 웹 페이지의 HTML 구조와 스크린샷을 찍어 AI에게 전달합니다.
- **생각(Think)**: AI가 사용자의 목표(예: "가장 저렴한 항공권 예약해줘")와 현재 페이지 상태를 분석합니다.
- **행동(Act)**: 분석 결과를 바탕으로 클릭, 타이핑 등 구체적인 브라우저 액션을 실행합니다.

---

### 💡 CUA(Computer-Use Agent)의 일종

우리가 쓸 `Browser-use`가 바로 이 **CUA(컴퓨터 사용 에이전트)** 개념을 웹 브라우저 환경에 특화하여 구현한 것입니다.
*"다나와 가서 RTX 4090 최저가 모델 3개만 찾아와"* 라고 자연어로 지시하면, AI가 알아서 브라우저를 켜고, 검색창을 찾아 클릭하고, 검색어를 입력한 뒤 최적의 결과를 추출해 옵니다.

---

### 🌐 Chrome DevTools Protocol (CDP) 브라우저 세션 공유 아키텍처

웹 에이전트 시스템을 구축할 때 가장 흔히 겪는 치명적인 문제는 **도구 간 브라우저 세션(쿠키) 단절**입니다.

> ⚠️ **문제 상황: 브라우저를 따로 띄우면?**
> - `browser-use`가 복잡한 폼을 채우고 **로그인을 성공**했습니다.
> - 하지만 뒤이어 데이터를 정밀 추출하려는 `Playwright` 도구(Level 1/2)가 새로운 브라우저 프로세스를 띄우면, **로그인 쿠키와 세션이 없어서 다시 로그인 화면으로 튕겨버립니다.**

이 문제를 해결하기 위해 AAWS는 **Chrome DevTools Protocol(CDP)** 기반의 **단일 브라우저 인스턴스 공유 아키텍처**를 적용했습니다.

```
                      ┌──────────────────────────────────────┐
                      │    단 1개의 Chrome 인스턴스 (PID: XXX) │
                      │    --remote-debugging-port=9242      │
                      └──────────────────┬───────────────────┘
                                         │ (CDP Port: 9242)
                    ┌────────────────────┴────────────────────┐
                    ▼                                         ▼
         [Playwright 도구들]                          [browser-use 도구]
     (connect_over_cdp 연결)                       (Browser(cdp_url=...) 연결)
   - extract_dom_skeleton                        - browse_web (자율 에이전트)
   - get_page_section                            - 마우스 클릭 / 로그인 수행
   - verify_selectors
```

#### 🔧 어떻게 동작하나요?
1. **CDP (Chrome DevTools Protocol)란?**
   - 크롬 브라우저가 외부 프로그램과 통신하기 위해 열어두는 **표준 원격 디버깅 포트(`9242`)**입니다.
2. **단일 프로세스 기동**: 
   - `PlaywrightManager` 싱글턴이 `--remote-debugging-port=9242` 옵션으로 **단 1개의 Chrome 프로세스**만 띄웁니다.
3. **다중 클라이언트 접속**:
   - **`browser-use` (`browse_web`)**: `Browser(cdp_url="http://localhost:9242")`로 연결하여 시각적 판단 및 로그인 자율 조작을 수행합니다.
   - **`Playwright 도구들` (L1/L2)**: `playwright.chromium.connect_over_cdp(...)`로 **동일한 크롬 포트**에 연결합니다.

#### ✨ CDP 공유의 3가지 핵심 이점
1. **🔑 완벽한 세션/쿠키 연속성 (Session Continuity)**: `browser-use`가 로그인해 둔 상태 그대로 L1/L2 도구가 즉시 내부 DOM을 추출할 수 있습니다.
2. **⚡ 시스템 리소스(RAM) 대폭 절약**: 도구마다 브라우저를 2~3개씩 띄우지 않고 1개의 크롬 메모리만 점유합니다.
3. **👁️ noVNC 실시간 시각화 통합**: VNC 화면(포트 `6080`)을 켜두면, 자율 로그인부터 DOM 추출까지 모든 과정이 **단 하나의 브라우저 창에서 매끄럽게 연속적으로 중계**됩니다.


In [ ]:
from app.tools.navigator import browse_web

# Level 3 자율 에이전트에게 자연어 미션 전달:
# quotes.toscrape.com에 접속하여 첫 번째 명언과 저자를 찾아달라고 요청

print("🚀 [Level 3] browse_web 자율 에이전트 가동 (OpenAI gpt-4.1-mini)...\n")
print("💡 HEADLESS=false 설정 시 브라우저가 자율적으로 움직이는 모습을 실시간으로 확인할 수 있습니다!")

browse_result = await browse_web.ainvoke({
    "task": "네이버 뉴스에 접속 후 세계면으로 들어간 후, 가장 눈에 띄는 기사 3개의 타이틀을 써봐.",
    "max_steps": 15
})

print("\n" + "=" * 60)
print("📋 [browse_web 실행 결과 보고]")
print("=" * 60)
print(browse_result)

In [ ]:
from app.tools.navigator import browse_web

# Level 3 자율 에이전트에게 자연어 미션 전달:
# quotes.toscrape.com에 접속하여 첫 번째 명언과 저자를 찾아달라고 요청

print("🚀 [Level 3] browse_web 자율 에이전트 가동 (OpenAI gpt-4.1-mini)...\n")
print("💡 HEADLESS=false 설정 시 브라우저가 자율적으로 움직이는 모습을 실시간으로 확인할 수 있습니다!")

browse_result = await browse_web.ainvoke({
    "task": "이 페이지의 첫 번째 명언(quote)의 본문과 저자 이름을 찾아서 알려줘.",
    "url": "http://quotes.toscrape.com",
    "max_steps": 15
})

print("\n" + "=" * 60)
print("📋 [browse_web 실행 결과 보고]")
print("=" * 60)
print(browse_result)

### 4.2 세션 공유 데모: browse_web 로그인 후 L1/L2 도구로 데이터 읽기

browser-use가 Chrome에서 로그인을 수행하면, **동일한 Chrome 프로세스를 공유하는 PlaywrightManager** 역시 로그인 세션(쿠키)을 즉시 공유받습니다.

아래 실습에서는:
1. `browse_web`으로 Quotes to Scrape 사이트에 로그인합니다.
2. 로그인 완료 후, Level 1 도구(`get_page_section`)로 로그인 사용자 전용 UI(예: 'Logout' 버튼)가 정상 노출되는지 확인합니다.

In [ ]:
# 1. browse_web을 통해 로그인 수행
login_task = "http://quotes.toscrape.com/login 페이지로 이동해서 아이디 'instructor_demo', 비밀번호 'pass1234'를 입력하고 로그인 버튼을 눌러줘."

print("🔐 [browse_web] 자율 로그인 수행 중...")
login_result = await browse_web.ainvoke({
    "task": login_task,
    "url": "http://quotes.toscrape.com/login",
    "max_steps": 15
})
print(login_result)

In [ ]:
# 2. 동일 Chrome 세션에서 Level 1 도구로 헤더 영역 확인 (Logout 링크 존재 여부 확인)
print("\n🍪 [Level 1 get_page_section] 로그인 세션 유지 확인...")
header_section = await get_page_section.ainvoke({
    "url": "http://quotes.toscrape.com",
    "root_selector": "div.header-box",
    "max_chars": 2000
})
print(header_section)

if "Logout" in header_section:
    print("\n🎉 [성공] browser-use가 로그인한 세션이 Playwright L1 도구에도 완벽하게 공유되었습니다!")
else:
    print("\nℹ️ 로그인 상태 확인 (HTML 내용 참조)")

In [ ]:
# 브라우저 및 리소스 정리
from app.tools.navigator import PlaywrightManager

manager = PlaywrightManager._instance
if manager:
    await manager.close()
    print("✅ 브라우저 종료 완료")

---
## 🎯 정리 및 핵심 요약 (Key Takeaways)

오늘 실습한 내용을 요약하면 다음과 같습니다.

1. **웹 데이터 수집의 적재적소(Best Fit) 도구 선택**
   - 대량 데이터 수집은 **`requests` + `BeautifulSoup`** 같은 경량 HTTP 루프가 가장 빠르고 저렴합니다.
   - 하지만 사이트 구조 분석과 동적 조작에는 **브라우저 제어 도구**가 반드시 결합되어야 합니다.

2. **3단계 에스컬레이션의 위력**
   - **Level 1 (DOM Skeleton & Selector Verify)**: 90% 이상의 일반 사이트에서 1~2초 만에 구조를 파악하고 수집 코드를 유도
   - **Level 2 (interact_page)**: AJAX 비동기 로딩, 더보기 클릭 등 경량 인터랙션을 해결
   - **Level 3 (browse_web)**: 로그인, CAPTCHA, 미지의 복잡한 SPA를 비전 기반으로 자율 해결

3. **CDP 세션 공유 아키텍처**
   - Chrome의 원격 디버깅 포트(`9242`)를 통해 Playwright와 browser-use가 하나의 브라우저를 공유함으로써,
     로그인 세션을 매끄럽게 인계받아 대량 수집 파이프라인으로 연결할 수 있습니다.

수고하셨습니다! 🚀